In [1]:
from dotenv import load_dotenv

load_dotenv
import os

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-4.1-mini",
    temperature=0.2,
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [3]:
result = llm.invoke("What is an agent?")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: OPENAI_A***********************************************************************************************************************************************************************Pt0A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
print(result)

In [ ]:
from langchain.tools import tool


@tool
def write_email(
    to: str,
    subject: str,
    content: str,
) -> str:
    """write and send an email"""
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

In [ ]:
type(write_email)

In [ ]:
write_email.args

In [ ]:
model_with_tool = llm.bind_tools(
    [write_email],
    tool_choice="any",
    parallel_tool_calls=False,
)
output = model_with_tool.invoke(
    "Draft a response to my boss (boss@company.ai) about tomorrow's meeting"
)

In [ ]:
type(output)

In [ ]:
output

In [ ]:
args = output.tool_calls[0]["args"]
args

In [ ]:
result = write_email.invoke(args)

In [ ]:
result

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class state_schema(TypedDict):
    request: str
    email: str


workflow = StateGraph(state_schema)

In [ ]:
def write_email_node(state: state_schema) -> state_schema:
    output = model_with_tool.invoke(state["request"])
    args = output.tool_calls[0]["args"]
    email = write_email.invoke(args)
    return {"email": email}

In [ ]:
workflow = StateGraph(state_schema)
workflow.add_node("write_email_node", write_email_node)
workflow.add_edge(START, "write_email_node")
workflow.add_edge("write_email_node", END)

app = workflow.compile()

In [ ]:
app

In [ ]:
app.invoke(
    {
        "request": "Draft a response to my boss (boss@company.ai) about tomorrow's meeting"
    }
)

In [ ]:
from typing import Literal
from langgraph.graph import MessagesState


def call_llm(state: MessagesState) -> MessagesState:
    """Run llm"""
    output = model_with_tool.invoke(state["messages"])
    return {"messages": [output]}


def run_tool(state: MessagesState):
    """perform the tool call"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        observation = write_email.invoke(tool_call["args"])
        result.append(
            {"role": "tool", "content": observation, "tool_call_id": tool_call["id"]}
        )
    return {"messages": result}


def should_continue(state: MessagesState) -> Literal["run_tool", "__end__"]:
    """Route to tool handler, or end if Done tool called"""
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "run_tool"
    return END


workflow = StateGraph(MessagesState)
workflow.add_node("call_llm", call_llm)
workflow.add_node("run_tool", run_tool)
workflow.add_edge(START, "call_llm")
workflow.add_conditional_edges(
    "call_llm", should_continue, {"run_tool": "run_tool", END: END}
)
workflow.add_edge("run_tool", END)
app=workflow.compile()

In [ ]:
app

In [ ]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!",
            }
        ]
    }
)
for m in result["messages"]:
    m.pretty_print()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[write_email],
    system_prompt="Respond to the user's request using the tools provided.",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Draft a response to my boss (boss@company.ai) "
                    "confirming that I want to attend Interrupt!"
                ),
            }
        ]
    }
)

for m in result["messages"]:
    m.pretty_print()

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[write_email],
    system_prompt="Respond to the user's request using the tools provided.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are some good practices for writing emails?",
            }
        ]
    },
    config,
)

In [ ]:
config = {"configurable": {"thread_id": "1"}}
state = agent.get_state(config)
for message in state.values["messages"]:
    message.pretty_print()

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Good, let's use lesson 3 to craft a response to my boss confirming that I want to attend Interrupt",
            }
        ]
    },
    config,
)
for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I like this, let's write the email to boss@company.ai",
            }
        ]
    },
    config,
)
for m in result["messages"]:
    m.pretty_print()

Interrupts

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver


class State(TypedDict):
    input: str
    user_feedback: str


def step_1(state):
    print("---Step 1---")
    pass


def human_feedback(state):
    print("---human_feedback---")
    feedback = interrupt("Please provide feedback:")
    return {"user_feedback": feedback}


def step_3(state):
    print("---Step 3---")
    pass


builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("human_feedback", human_feedback)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "human_feedback")
builder.add_edge("human_feedback", "step_3")
builder.add_edge("step_3", END)

# Set up memory
memory = InMemorySaver()

# Add
graph = builder.compile(checkpointer=memory)

In [ ]:
graph

In [ ]:
# Input
initial_input = {"input": "hello world"}

# Thread
thread = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="updates"):
    print(event)
    print("\n")

In [ ]:
# Continue the graph execution
for event in graph.stream(
    Command(resume="go to step 3!"),
    thread,
    stream_mode="updates",
):
    print(event)
    print("\n")